In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import WikiCS
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
from torch_geometric.nn import SAGEConv, Node2Vec
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataset = WikiCS(root="data/WikiCS")
data = dataset[0]

Using device: cuda


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\datasets\wikics.py:45: UserWarning: The WikiCS dataset now returns an undirected graph by default. Please explicitly specify 'is_undirected=False' to restore the old behavior.
  warnings.warn(
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.

In [2]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False
)

train_data, val_data, test_data = transform(data)

train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

In [7]:
# DeepWalk is the same as Node2Vec with p=1 and q=1

dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=10,
    context_size=7,
    walks_per_node=10,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
).to(device)

loader = dw_model.loader(batch_size=256, shuffle=True)
dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)


In [8]:
def train_deepwalk():

    dw_model.train()
    total_loss = 0

    for pos_rw, neg_rw in loader:

        dw_optimizer.zero_grad()

        loss = dw_model.loss(pos_rw.to(device), neg_rw.to(device))

        loss.backward()
        dw_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_deepwalk(data):

    dw_model.eval()

    z = dw_model()
    z = F.normalize(z, p=2, dim=1)

    out = (z[data.edge_label_index[0]] * z[data.edge_label_index[1]]).sum(dim=-1).sigmoid()

    y_true = data.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()

    auc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)

    return auc, ap


In [9]:
for epoch in range(1, 21):

    loss = train_deepwalk()

    if epoch % 5 == 0:

        val_auc, val_ap = evaluate_deepwalk(val_data)

        print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Val AUC {val_auc:.4f} | Val AP {val_ap:.4f}")
        dw_auc, dw_ap = evaluate_deepwalk(test_data)

        print("AUC:", dw_auc)
        print("AP :", dw_ap)

Epoch 005 | Loss 2.0559 | Val AUC 0.7891 | Val AP 0.7913
AUC: 0.7914785409196616
AP : 0.7910244067245242
Epoch 010 | Loss 1.2056 | Val AUC 0.9189 | Val AP 0.9304
AUC: 0.9230757065132926
AP : 0.9325772677973303
Epoch 015 | Loss 1.0465 | Val AUC 0.9528 | Val AP 0.9606
AUC: 0.9563931906921419
AP : 0.9631801044818503
Epoch 020 | Loss 1.0041 | Val AUC 0.9621 | Val AP 0.9685
AUC: 0.9657003705141292
AP : 0.9711208906014115


In the above cell after running a few times the training with different configurations and monitoring the outputs I picked the best number of epochs and configuration.

In [11]:
print("\nTraining DeepWalk")
num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):

    dw_model = Node2Vec(
    train_data.edge_index,
    embedding_dim=64,
    walk_length=10,
    context_size=7,
    walks_per_node=10,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True
    ).to(device)

    loader = dw_model.loader(batch_size=256, shuffle=True)
    dw_optimizer = torch.optim.SparseAdam(dw_model.parameters(), lr=0.01)
    for epoch in range(1, 21):
        loss = train_deepwalk()
        
    dw_auc, dw_ap = evaluate_deepwalk(test_data)
    test_aucs.append(dw_auc)
    test_aps.append(dw_ap)
    print(f"Run {run:02d} Completed - Test AUC: {dw_auc:.4f}, Test AP: {dw_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")



Training DeepWalk
Run 01 Completed - Test AUC: 0.9658, Test AP: 0.9716
Run 02 Completed - Test AUC: 0.9657, Test AP: 0.9712
Run 03 Completed - Test AUC: 0.9655, Test AP: 0.9710
Run 04 Completed - Test AUC: 0.9659, Test AP: 0.9715
Run 05 Completed - Test AUC: 0.9661, Test AP: 0.9717
Run 06 Completed - Test AUC: 0.9661, Test AP: 0.9716
Run 07 Completed - Test AUC: 0.9658, Test AP: 0.9715
Run 08 Completed - Test AUC: 0.9661, Test AP: 0.9716
Run 09 Completed - Test AUC: 0.9658, Test AP: 0.9716
Run 10 Completed - Test AUC: 0.9655, Test AP: 0.9711
Run 11 Completed - Test AUC: 0.9656, Test AP: 0.9711
Run 12 Completed - Test AUC: 0.9653, Test AP: 0.9711
Run 13 Completed - Test AUC: 0.9657, Test AP: 0.9712
Run 14 Completed - Test AUC: 0.9663, Test AP: 0.9717
Run 15 Completed - Test AUC: 0.9655, Test AP: 0.9713
Run 16 Completed - Test AUC: 0.9655, Test AP: 0.9712
Run 17 Completed - Test AUC: 0.9648, Test AP: 0.9704
Run 18 Completed - Test AUC: 0.9658, Test AP: 0.9711
Run 19 Completed - Test AUC